# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FawadAhmad-bilal/flyrank-assignment-1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Given a content page's visibility, freshness, and search-position signals, which pages are most likely to be declining in impressions right now — and can an editorial team use that ranking to decide what to review first, this week, without reading every page in the portfolio?

**Decision this supports:** a weekly content-review shortlist for an SEO/content team, not an automated rewrite or de-indexing system (see Limitations, Section 5, and the no-go list in work/notebooks/w07_action_playbook.ipynb).

**Release used:** the starter dataset shipped in this repo — `data/raw/content_refresh_anonymized.csv`, 30,000 rows, one row per pseudonymized content item, 32 pseudonymized clients, trailing-90-day metrics, single snapshot (not the full ~79M-row warehouse). **Excluded:** `trend_direction` and `trend_pct` from the feature set (they define the label, see Methodology); `provider_used` / `model_used` (high missingness, not predictive of the question asked); rows are never dropped by activity level — low-visibility rows are scored `insufficient_data` rather than excluded, to avoid a silent survivorship filter. **Public-safe:** all identifiers used here are the dataset's own pseudonyms (`content_id`, `client_id`); no real client names, URLs, or search queries appear anywhere in this repo.

In [1]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows, {df['client_id'].nunique()} pseudonymized clients")
print(df[["content_type","main_intent"]].describe(include="all").loc[["count","unique","top","freq"]])

30,000 rows, 32 pseudonymized clients
           content_type    main_intent
count             30000          27626
unique                3              4
top     keyword article  informational
freq              27207          17235


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining_label = 1` if `trend_direction == "down"` (FlyRank's own definition: 30-day-vs-previous-30-day impression change, >10% decline). **Features (all knowable at prediction time):** `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `engagement_rate`, `ai_traffic_pct`, `content_age_days`, `days_since_last_update`, `word_count`, `content_type`, `main_intent`. **Baseline:** a transparent rule (Week 4) scoring staleness magnitude + CTR-shortfall against a position-tier benchmark, no fitted weights. **Models:** Logistic Regression, then Random Forest (300 trees, depth 8) — chosen per the training-honest-models toolkit for a yes/no label evaluated as a ranking. **Validation:** `GroupShuffleSplit` by `client_id` (75/25, seed 42) — not random, because random splitting let the same client appear in both train and test and inflated scores by memorization (see below). **Leakage checks:** `trend_direction`/`trend_pct` excluded from every feature set; verified by deliberately re-adding `trend_pct` and confirming precision@K jumps to a perfect 1.000 (full detail: `w06_validation_audit.ipynb`) — proof the exclusion actually matters, not just a stated rule.

In [2]:
import json
m = json.load(open("../outputs/paper_results_metrics.json"))
print("Random-split RF (naive) vs grouped-split RF (honest), precision@20:")
print(f"  random split : {m['random_forest_random_split'][0]:.3f}")
print(f"  grouped split: {m['random_forest_grouped'][0]:.3f}")
print("  -> gap of", round(m['random_forest_random_split'][0]-m['random_forest_grouped'][0],3), "is memorization, not skill")

Random-split RF (naive) vs grouped-split RF (honest), precision@20:
  random split : 0.900
  grouped split: 0.550
  -> gap of 0.35 is memorization, not skill


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Same held-out clients, same metric, three scorers. Base rate: 0.517.

| k | Baseline | Logistic Regression | Random Forest |
|---|---|---|---|
| 20 | 0.600 | **0.800** | 0.550 |
| 50 | 0.520 | **0.660** | 0.600 |
| 100 | 0.550 | 0.580 | **0.620** |
| 500 | 0.514 | 0.530 | **0.626** |

Logistic Regression wins at the top of the queue; Random Forest pulls ahead at greater depth. Neither wins everywhere — reported honestly rather than picking the metric that flatters one model.

In [3]:
print(json.dumps(m, indent=2))

{
  "ks": [
    20,
    50,
    100,
    500
  ],
  "base_rate": 0.517,
  "baseline": [
    0.6,
    0.52,
    0.55,
    0.514
  ],
  "logistic_regression": [
    0.8,
    0.66,
    0.58,
    0.53
  ],
  "random_forest_grouped": [
    0.55,
    0.6,
    0.62,
    0.626
  ],
  "random_forest_random_split": [
    0.9,
    0.9,
    0.91,
    0.852
  ]
}


## 5. Limitations

*What this work cannot claim.*

- **Scope:** 30,000 rows / 32 clients — a teaching slice, not the full warehouse. Archetype boundaries and precision numbers may not hold at 79M-row scale.
- **Snapshot, not longitudinal:** one 90-day window. No evidence here about seasonal effects or how quickly a page's risk changes month to month.
- **Label noise at low volume:** a percentage-based decline label is unstable on a near-zero base — several of the Week-5 false negatives had 1-2 impressions total, where a single-session swing reads as a 100% change. This is a property of the label, not something more data fixes.
- **Correlational, not causal:** nothing here supports "refreshing X will cause Y" — only "pages that look like X tend to also show Y", per the writing-honest-claims skill's claim ladder.
- **Client generalization is untested beyond the 8 held-out clients** in this run; a client with a very different content mix (e.g., mostly navigational pages) may not fit these patterns.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Full detail in `w07_action_playbook.ipynb`. Four content archetypes (K-Means, k=4, visible pages only), each mapped to a default action — with the model's raw score overridden by archetype membership where domain judgment says the score alone would mislead (see the Early-Stage rule):

1. **Refresh first: High-Traffic Long-Form, Overdue** (2,745 pages) — smallest group, largest audience reach per page touched.
2. **Refresh soon: Aging & Under-Refreshed** (5,423 pages) — the textbook stale-content case.
3. **Leave alone: Early-Stage / Still Settling** (8,130 pages) — best position, freshest, but flagged high-risk by the raw model; treated as early-lifecycle noise, not a real problem, until they age past ~90 days.
4. **Maintain current cadence: Refreshed Veterans** (5,708 pages) — oldest pages, but the lowest risk once already kept fresh — independent confirmation of FlyRank's own paper finding that refreshing old content works.

In [4]:
queue_metrics = json.load(open("../outputs/playbook_metrics.json"))
print("Action mix:", queue_metrics["action_counts"])

Action mix: {'monitor_dont_touch': 8130, 'insufficient_data': 7994, 'monitor': 5108, 'refresh_soon': 3754, 'refresh_priority': 3042, 'no_action': 1972}


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Figures embedded in the deployed paper (`docs/assets/`): `results_vs_baseline.svg` (model vs baseline table as a chart), `split_gap.svg` (the random-vs-grouped-split leakage story), `archetype_mix.svg`, `action_mix.svg`. All regenerated by the code below — rerun this cell to reproduce them exactly.

In [5]:
# All four figures were generated in the working sessions for w05/w06/w07 and this notebook,
# and are committed at docs/assets/. To regenerate from scratch, rerun:
#   - the Results section 4 cell above (produces results_vs_baseline.svg, split_gap.svg)
#   - w07_action_playbook.ipynb section 5 (produces archetype_mix.svg, action_mix.svg)
import os
for f in ["results_vs_baseline.svg", "split_gap.svg", "archetype_mix.svg", "action_mix.svg"]:
    path = f"../../docs/assets/{f}"
    print(f, "OK" if os.path.exists(path) else "MISSING")

results_vs_baseline.svg OK
split_gap.svg OK
archetype_mix.svg OK
action_mix.svg OK


## Closing: repurposed cuts

**5-minute demo outline:** (1) Open with the question — "which of 30,000 pages should an editor look at first?" (2) Show the baseline rule, then the model-vs-baseline chart — name the honest result (LR wins at the top, RF wins deeper). (3) Show the split-gap chart — this is the moment that earns credibility: "here's what happens if you validate it the easy way instead." (4) Show one archetype (Refreshed Veterans) and connect it back to FlyRank's own paper finding. (5) Close on the no-go list — what this system explicitly refuses to automate.

**Social-post cut:** "Trained a model to flag declining content pages for FlyRank's SEO dataset. The interesting part wasn't the model — it was catching that my Random Forest's 90% precision was mostly client memorization. Under an honest client-holdout split, it drops to 55%. Full writeup + repo: [link]."

**3-sentence employer summary:** Built and validated a content-decline prediction pipeline on a 30,000-row, 32-client SEO dataset, comparing a transparent rule-based baseline against Logistic Regression and Random Forest models. Caught and fixed a validation flaw where a naive random split overstated Random Forest's precision by 35 points due to client-level memorization, using a grouped holdout split instead. Delivered a ranked, archetype-based content-review playbook with explicit human-review and no-automation boundaries, deployed as a public research page.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
